**Archived legacy method.** Set `YEAR` and, only when intentionally writing disposable outputs, set `SAVE = True`. Outputs go to `notebooks/outputs/<YEAR>/`; this notebook does not publish canonical pipeline data.

In [ ]:
# Legacy notebook controls — edit only these values
YEAR = 2026
SAVE = False

from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RAW_ROOT = REPO_ROOT / "data" / "raw"
PARTITION_ROOT = REPO_ROOT / "data" / "partitions"
PRODUCT_ROOT = REPO_ROOT / "data" / "products"
OUTPUT_ROOT = REPO_ROOT / "notebooks" / "outputs" / str(YEAR)
GEODATA_OUTPUT_ROOT = OUTPUT_ROOT / "geodata"

DEPARTMENTS_PATH = RAW_ROOT / "demographics" / "departments.csv"
DEPARTMENT_STATS_PATH = RAW_ROOT / "demographics" / "departmental_stats_2023.csv"
ARRONDISSEMENT_STATS_PATH = RAW_ROOT / "demographics" / "arrondissement_stats_2023.csv"
DEPARTMENTS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "departments.geojson"
REGIONS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "regions.geojson"
ARRONDISSEMENTS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "arrondissements-avec-outre-mer.geojson"
PARIS_GEOMETRY_PATH = RAW_ROOT / "geodata" / "paris_arrondissements.geojson"
MONACO_GEOMETRY_PATH = RAW_ROOT / "geodata" / "monaco.geojson"


def annual_restaurant_product(year):
    candidates = [
        PRODUCT_ROOT / "france" / str(year) / "all_restaurants(arrondissements).csv",
        PRODUCT_ROOT / "france" / f"all_restaurants(arrondissements)_{year % 100:02d}.csv",
        REPO_ROOT / "Years" / str(year) / "data" / "France" / "all_restaurants(arrondissements).csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"No accepted annual restaurant product for {year}: {candidates}")


def save_csv(frame, filename):
    target = OUTPUT_ROOT / filename
    if not SAVE:
        print(f"SAVE=False: not writing {target}")
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(target, index=False)
    print(f"Wrote {target}")


def save_geojson(frame, filename):
    target = GEODATA_OUTPUT_ROOT / filename
    if not SAVE:
        print(f"SAVE=False: not writing {target}")
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    frame.to_file(target, driver="GeoJSON")
    print(f"Wrote {target}")


# Michelin Restaurants in France - *Arrondissements*

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

Importing restaurant data

In [ ]:
france_data = pd.read_csv(PRODUCT_ROOT / "france" / str(YEAR) / "all_restaurants.csv")
france_data.head()

In [ ]:
print(france_data.columns.tolist())

----
&nbsp;
Demographics data from [INSEE](https://statistiques-locales.insee.fr/#c=indicator&view=map5)

In [ ]:
stats_locale = pd.read_csv(ARRONDISSEMENT_STATS_PATH, sep=';', header=None)
# Use the third row as column headers
stats_locale.columns = stats_locale.iloc[2]
# Drop the metadata rows
stats_locale = stats_locale.drop([0, 1, 2]).reset_index(drop=True)
print(stats_locale.columns.tolist())

In [ ]:
stats_locale = stats_locale.rename(columns={
    'Code': 'code',
    'Libellé': 'arrondissement',
    'Taux de pauvreté 2021': 'poverty_rate(%)',
    'Salaire net horaire moyen 2022': 'average_net_hourly_wage(€)',
    'Population municipale 2022': 'municipal_population',
    'Densité de population (historique depuis 1876) 2021': 'population_density(inhabitants/sq_km)',
})
stats_locale = stats_locale[['code', 'arrondissement', 'municipal_population', 'population_density(inhabitants/sq_km)',
                             'poverty_rate(%)', 'average_net_hourly_wage(€)']]
stats_locale.head()

In [ ]:
print(stats_locale.columns.tolist())

Loading in [GeoJSON data](https://github.com/gregoiredavid/france-geojson/blob/master/arrondissements-avec-outre-mer.geojson)

In [ ]:
# Load the GeoJSON file
geoJSON_df = gpd.read_file(ARRONDISSEMENTS_GEOMETRY_PATH)
geoJSON_df.head()

In [ ]:
print(geoJSON_df.columns.tolist())

----
&nbsp;
### Removing *outre-mers*

In [ ]:
# We leverage the format of the first two digits of the 'code' column
# where 01 - 95 represents mainland france
geoJSON_df = geoJSON_df.sort_values(by='code')
geoJSON_df = geoJSON_df[~geoJSON_df['code'].str.startswith('97')]
geoJSON_df = geoJSON_df.reset_index(drop=True)
geoJSON_df.shape

In [ ]:
# We again remove '97' leveraging a pattern
stats_locale = stats_locale.sort_values(by='code')
stats_locale = stats_locale[~stats_locale['code'].str.startswith('97')]
stats_locale = stats_locale.reset_index(drop=True)
stats_locale.shape

It's good news the dataframes have the same length

----
&nbsp;
### Looking for discrepancies in the `nom`, `arrondissement` columns

In [ ]:
set1 = set(stats_locale['arrondissement'].unique())
set2 = set(geoJSON_df['nom'].unique())
print(f"Sets equal? {set1 == set2}")

In [ ]:
print("In set1 but not in set2: ", set1 - set2)
print("\nIn set2 but not in set1: ", set2 - set1)

We will add articles to the GeoDataFrame by creating a reverse mapping dictionary.

In [ ]:
def clean_name(name):
    for prefix in ['Le ', 'La ', 'Les ', "L'"]:
        if name.startswith(prefix):
            name = name[len(prefix):]
    return name.strip()

def match_articleless_to_full(articleless_set, full_set):
    """
    Returns a mapping from articleless names (e.g. 'Mans') to full names (e.g. 'Le Mans')
    """
    # Build a reverse map: cleaned name → full name with article
    cleaned_to_full = {clean_name(name): name for name in full_set}

    # Match articleless name to full name via cleaned lookup
    mapping = {}
    for name in articleless_set:
        full_name = cleaned_to_full.get(name)
        if full_name:
            mapping[name] = full_name

    return mapping

In [ ]:
# set1_clean = set(map(clean_name, stats_locale['arrondissement'].unique()))
# set2_clean = set(map(clean_name, geoJSON_df['nom'].unique()))

reversed_map = match_articleless_to_full(set2, set1)

# Update 'stats_locale' with the new names
# stats_locale['arrondissement'] = stats_locale['arrondissement'].apply(reversed_map)
geoJSON_df['nom'] = geoJSON_df['nom'].map(
    lambda x: reversed_map.get(clean_name(x), x)
)

In [ ]:
set1 = set(stats_locale['arrondissement'].unique())   # Full names
set2 = set(geoJSON_df['nom'].unique())
# Check for discrepancies again
print("In set1 but not in set2: ", set1 - set2)
print("\nIn set2 but not in set1: ", set2 - set1)

In [ ]:
# Correct the discrepancy manually
if 'Val-de-Briey' in set1 and 'Briey' in set2:
    geoJSON_df['nom'] = geoJSON_df['nom'].replace('Briey', 'Val-de-Briey')

In [ ]:
article_mask = stats_locale['arrondissement'].str.startswith(('Le ', 'La ', 'Les ', "L'"))
print("Rows with definite articles:", article_mask.sum())
print("Total rows:", len(stats_locale))

In [ ]:
article_mask = geoJSON_df['nom'].str.startswith(('Le ', 'La ', 'Les ', "L'"))
print("Rows with definite articles:", article_mask.sum())
print("Total rows:", len(geoJSON_df))

----
&nbsp;
### Mapping individual restaurants to *arrondissements* based on (latitude, longitude) Geospatial data

In [ ]:
# Convert france_data to GeoDataFrame
geometry = [Point(xy) for xy in zip(france_data.longitude, france_data.latitude)]
france_geo = gpd.GeoDataFrame(france_data, geometry=geometry)

In [ ]:
france_geo.head()

In [ ]:
france_geo.crs = "EPSG:4326"

In [ ]:
joined_data = gpd.sjoin(france_geo, geoJSON_df[['nom', 'geometry']], predicate='within', how='left')
joined_data.rename(columns={'nom': 'arrondissement'}, inplace=True)

In [ ]:
# Convert the GeoDataFrame to a normal DataFrame
france_df = pd.DataFrame(joined_data)

# Drop the 'geometry' and 'index_right' columns
france_df = france_df.drop(columns=['geometry', 'index_right'])
print(france_df.columns.tolist())

In [ ]:
new_order = ['name', 'address', 'location', 'arrondissement', 'department_num',
             'department', 'capital', 'region', 'price', 'cuisine', 'url',
             'award', 'stars', 'greenstar', 'longitude', 'latitude']
france_df = france_df[new_order]
france_df.head(10)

In [ ]:
france_df.info()

In [ ]:
# Unique arrondissement in the dataset
france_df.arrondissement.nunique()

----
&nbsp;
### Adding Parisien *arrondissement municipaux*

In [ ]:
all_france = france_df.copy()

In [ ]:
import requests
from io import StringIO

url = "https://en.wikipedia.org/wiki/Arrondissements_of_Paris"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()

tables = pd.read_html(StringIO(response.text))
print(f"Found {len(tables)} tables")

In [ ]:
# The table of interest is found by inspection
paris_arr = tables[1]
paris_arr.head()

In [ ]:
print(paris_arr.columns.tolist())

In [ ]:
# Drop specified columns from the DataFrame
paris_arr = paris_arr.drop(columns=['Coat of arms', 'Area (km2)', 'Population (2017 estimate)', 'Density (2017) (inhabitants per km2)', 'Peak of population', 'Mayor (2020–2026)'])

paris_arr = paris_arr.rename(columns={'Arrondissement (R for Right Bank, L for Left Bank)': 'arrondissement', 'Name': 'name'})

In [ ]:
# Extract the first part of the 'arrondissement' using regex
paris_arr['arrondissement'] = paris_arr['arrondissement'].str.extract(r'(\d+\w{2})')
paris_arr.head()

Search `france_data` for 'Paris'

In [ ]:
paris_rows = all_france[all_france['location'].str.contains('Paris', case=False, na=False)]
paris_rows.head()

In [ ]:
# Mask to identify rows with 'arrondissement' == 'Paris'
mask_paris = all_france['arrondissement'] == 'Paris'

# Extract the last two digits from the postal code for the mask
all_france.loc[mask_paris, 'postal_digits'] = all_france.loc[mask_paris, 'location'].str[-2:].astype(int)

# Overwrite the 'arrondissement' column for the mask
all_france.loc[mask_paris, 'arrondissement'] = all_france.loc[mask_paris, 'postal_digits']

# Drop the temporary 'postal_digits' column
all_france.drop(columns=['postal_digits'], inplace=True, errors='ignore')  # Using errors='ignore' to ensure code doesn't break if column doesn't exist

In [ ]:
all_france.head()

In [ ]:
# Filter out Paris rows
paris_rows = all_france[all_france['department'].str.contains('Paris', case=False, na=False)].copy()

In [ ]:
# Extract numbers from "1st", "2nd", etc. in paris_arr to a new temporary column
paris_arr['arr_num'] = paris_arr['arrondissement'].str.extract(r'(\d+)').astype(float)

# Create a dictionary that maps these numbers to the tuple (arrondissement, name)
paris_tuple_dict = dict(zip(paris_arr['arr_num'], zip(paris_arr['arrondissement'], paris_arr['name'])))
paris_tuple_dict = {k: f"{v[0]} ({v[1]})" for k, v in paris_tuple_dict.items()}

In [ ]:
# Map the arrondissement values in all_france to the paris_tuple_dict
new_arr_values = all_france['arrondissement'].map(paris_tuple_dict)

# Fill NaN values with existing values from arrondissement column
new_arr_values.fillna(all_france['arrondissement'], inplace=True)

In [ ]:
# Assign the new series to the arrondissement column
all_france['arrondissement'] = new_arr_values
all_france.head()

In [ ]:
print(all_france.region.unique())

In [ ]:
# all_france has some English region names -> need to map them to French
region_mapping = {
    'Brittany': 'Bretagne',
    'Corsica': 'Corse',
    'Normandy': 'Normandie'
}

# Map the English region names to the French ones used in df
all_france['region'] = all_france['region'].replace(region_mapping)

In [ ]:
print(all_france.region.unique())

#### Exporting as `csv` file

In [ ]:
# Export the DataFrame to a .csv file
save_csv(all_france, "all_restaurants(arrondissements).csv")

----
&nbsp;
## Subsetting for Paris
#### Creating `paris_restaurants.geojson`

In [ ]:
paris = all_france[all_france['department'].str.contains('Paris', case=False, na=False)].copy()
print(f"Shape: {paris.shape}")
paris.head()

In [ ]:
print(f"Columns:\n{paris.columns}")

In [ ]:
print(f"No unique arrondissements: {len(paris.arrondissement.unique())}\n{paris.arrondissement.unique()}")

----
&nbsp;
### Adding Parisien shape files

In [ ]:
# Create a subset of the data based on unique arrondissements and relevant columns
paris_subset = paris[['location', 'arrondissement', 'department_num', 'department', 'capital', 'region']].drop_duplicates()

# Display the unique arrondissements and subsetted data
print(f"Unique Arrondissements: {paris_subset['arrondissement'].nunique()}")
paris_subset.head()

In [ ]:
# Load the GeoJSON file
geoJSON_paris = gpd.read_file(PARIS_GEOMETRY_PATH)
print(f"Shape: {geoJSON_paris.shape}")
geoJSON_paris.head()

In [ ]:
print(f"Columns: {geoJSON_paris.columns.tolist()}")

In [ ]:
print(f"No unique arrondissements: {len(geoJSON_paris.l_ar.unique())}\n{geoJSON_paris.l_ar.unique()}")

#### Creating a new numerical column in both `paris_subset` and `geoJSON_paris`

In [ ]:
# Extract the numerical part from the 'arrondissement' column in paris_subset
paris_subset['arrondissement_num'] = paris_subset['arrondissement'].str.extract(r'(\d+)').astype(int)

# Extract the numerical part from the 'l_ar' column in geoJSON_paris
geoJSON_paris['arrondissement_num'] = geoJSON_paris['l_ar'].str.extract(r'(\d+)').astype(int)

In [ ]:
# Get unique arrondissement numbers from both
paris_arrondissements = set(paris_subset['arrondissement_num'].unique())
geojson_arrondissements = set(geoJSON_paris['arrondissement_num'].unique())

# Find out which arrondissements are missing in geoJSON
missing_arrondissements = geojson_arrondissements - paris_arrondissements
print(f"Missing Arrondissements: {missing_arrondissements}")

In [ ]:
# Convert the 'arrondissement_num' column to int, ignoring errors in case of any non-convertible values
paris_subset['arrondissement_num'] = pd.to_numeric(paris_subset['arrondissement_num'], errors='coerce')
paris_subset = paris_subset.sort_values(by='arrondissement_num').reset_index(drop=True)
paris_subset.tail()

In [ ]:
# Merge the two DataFrames on 'arrondissement_num'
paris_geo_merged = paris_subset.merge(
    geoJSON_paris[['arrondissement_num', 'geometry']], on='arrondissement_num', how='left'
)

In [ ]:
paris_geo_merged = paris_geo_merged.drop(columns=['arrondissement_num'])
paris_geo_merged = paris_geo_merged.drop(index=15)
print(f"Shape: {paris_geo_merged.shape}")
paris_geo_merged.tail()

In [ ]:
paris_geo_merged

#### Adding restaurant counts and locations

In [ ]:
print(f"Parisien Restos: {paris.shape}")
paris.head()

In [ ]:
paris_data_copy = paris.copy()

# Create dummy variables for each category of 'star'
paris_data_copy['green_stars'] = paris_data_copy['greenstar'].apply(lambda x: 1 if x == 1 else 0)
paris_data_copy['selected'] = paris_data_copy['stars'].apply(lambda x: 1 if x == 0.25 else 0)
paris_data_copy['bib_gourmand'] = paris_data_copy['stars'].apply(lambda x: 1 if x == 0.5 else 0)
paris_data_copy['1_star'] = paris_data_copy['stars'].apply(lambda x: 1 if x == 1.0 else 0)
paris_data_copy['2_star'] = paris_data_copy['stars'].apply(lambda x: 1 if x == 2.0 else 0)
paris_data_copy['3_star'] = paris_data_copy['stars'].apply(lambda x: 1 if x == 3.0 else 0)

In [ ]:
# Group by 'arrondissement' and sum 'bib', '1_star', '2_star' and '3_star'
paris_arrondissement_grouped = paris_data_copy.groupby('arrondissement')[['green_stars', 'selected', 'bib_gourmand', '1_star', '2_star', '3_star']].sum()

In [ ]:
# Create a 'total_ star' column - sum of stars
# Individual stars are summed here. ie, If 2 star then stars = 2
paris_arrondissement_grouped['total_stars'] = paris_arrondissement_grouped['1_star']*1 + paris_arrondissement_grouped['2_star']*2 + paris_arrondissement_grouped['3_star']*3

# Create a 'total' column = sum of restaurants
paris_arrondissement_grouped['starred_restaurants'] =  paris_arrondissement_grouped['1_star'] + paris_arrondissement_grouped['2_star'] + paris_arrondissement_grouped['3_star']

paris_arrondissement_grouped = paris_arrondissement_grouped.reset_index()
if 'index' in paris_arrondissement_grouped.columns:
    paris_arrondissement_grouped = paris_arrondissement_grouped.rename(columns={'index': 'arrondissement'})

In [ ]:
print(f"Shape: {paris_arrondissement_grouped.shape}")
paris_arrondissement_grouped.head()

In [ ]:
# Merge the grouped data with the geospatial data
paris_final_merged = paris_geo_merged.merge(paris_arrondissement_grouped, on='arrondissement', how='left')

#### Adding individual restaurants

In [ ]:
print(paris.columns.tolist())

In [ ]:
# Create a separate DataFrame with star ratings, regions, and coordinates
location_paris_arrond = paris[['stars', 'arrondissement', 'latitude', 'longitude']]

# Aggregate latitude and longitude into lists
grouped = location_paris_arrond.groupby(['stars', 'arrondissement']).agg({'latitude': list, 'longitude': list})

# Zip the lists of latitudes and longitudes into coordinates
grouped['coordinates'] = grouped.apply(lambda x: list(zip(x['latitude'], x['longitude'])), axis=1)

# Convert to dictionary
location_paris_arrond = grouped['coordinates'].to_dict()

In [ ]:
# Create a mapping from star values to string labels
star_label_mapping = {
    0.25: 'Selected',
    0.5: 'Bib',
    1: '1',
    2: '2',
    3: '3'
}

# Now create a function to map these dictionaries to original DataFrame
def map_locations_arrond(row):
    return {star_label_mapping[stars]: location_paris_arrond.get((stars, row['arrondissement'])) for stars in [0.25, 0.5, 1, 2, 3]}

paris_final_merged['locations'] = paris_final_merged.apply(map_locations_arrond, axis=1)

In [ ]:
# Drop 'Paris, ' prefix from 'location' and rename to 'code'
paris_final_merged['code'] = paris_final_merged['location'].str.replace('Paris, ', '')
paris_final_merged = paris_final_merged.drop(columns=['location'])

# Reorder columns so 'code' is first and 'geometry' is last
cols = ['code'] + [col for col in paris_final_merged.columns if col not in ['code', 'geometry']] + ['geometry']
paris_final_merged = paris_final_merged[cols]

In [ ]:
# Convert the DataFrame into a GeoDataFrame using the 'geometry' column
paris_final_merged_gdf = gpd.GeoDataFrame(paris_final_merged, geometry='geometry')

In [ ]:
print(f"Shape: {paris_final_merged_gdf.shape}")
print(f"Type: {type(paris_final_merged_gdf)}")
paris_final_merged_gdf.head()

#### Export Paris data as GeoJSON file

In [ ]:
save_geojson(paris_final_merged_gdf, "paris_restaurants.geojson")

----
&nbsp;
## Rest of France
#### Grouping by *arrondissement* based on number of Michelin stars

In [ ]:
france_data_copy = france_df.copy()

# Create dummy variables for each category of 'star'
france_data_copy['green_stars'] = france_data_copy['greenstar'].apply(lambda x: 1 if x == 1 else 0)
france_data_copy['selected'] = france_data_copy['stars'].apply(lambda x: 1 if x == 0.25 else 0)
france_data_copy['bib_gourmand'] = france_data_copy['stars'].apply(lambda x: 1 if x == 0.5 else 0)
france_data_copy['1_star'] = france_data_copy['stars'].apply(lambda x: 1 if x == 1.0 else 0)
france_data_copy['2_star'] = france_data_copy['stars'].apply(lambda x: 1 if x == 2.0 else 0)
france_data_copy['3_star'] = france_data_copy['stars'].apply(lambda x: 1 if x == 3.0 else 0)

#### By *`arrondissement`*
We sort `france_data` by total number of awarded restaurants

In [ ]:
# Group by 'arrondissement' and sum 'bib', '1_star', '2_star' and '3_star'
arrondissement_grouped = france_data_copy.groupby('arrondissement')[['green_stars', 'selected', 'bib_gourmand', '1_star', '2_star', '3_star']].sum()

# Create a copy for plotting
arrondissement_grouped_two = arrondissement_grouped.copy()

`arrondissement_grouped` is created to be merged with the demographics data

In [ ]:
# Create a 'total_ star' column - sum of stars
# Individual stars are summed here. ie, If 2 star then stars = 2
arrondissement_grouped['total_stars'] = arrondissement_grouped['1_star']*1 + arrondissement_grouped['2_star']*2 + arrondissement_grouped['3_star']*3

# Create a 'total' column = sum of restaurants
arrondissement_grouped['starred_restaurants'] =  arrondissement_grouped['1_star'] + arrondissement_grouped['2_star'] + arrondissement_grouped['3_star']

# Sort the dataframe by the 'total_stars' column in descending order
arrondissement_grouped.sort_values('total_stars', ascending=True, inplace=True)

In [ ]:
arrondissement_grouped.head()

----
&nbsp;
## Plotting starred restaurants by *`arrondissement`*

We use `arrondissement_grouped_two` for plotting whilst excluding Paris

In [ ]:
# Create a 'total_starred' column - sum of starred restaurants
arrondissement_grouped_two['total_starred'] = arrondissement_grouped_two['1_star'] + arrondissement_grouped_two['2_star'] + arrondissement_grouped_two['3_star']

# Sort the dataframes by the 'total' and 'total_stars' columns in descending order
arrondissement_grouped_two.sort_values('total_starred', ascending=True, inplace=True)

In [ ]:
# Exclude 'Paris'
arrondissement_grouped_no_paris = arrondissement_grouped_two[arrondissement_grouped_two.index.get_level_values('arrondissement') != 'Paris']

In [ ]:
# Sort by 'total_starred' in descending order and keep the top 10
arrondissement_grouped_top10 = arrondissement_grouped_no_paris.sort_values('total_starred', ascending=False).head(10)

# Drop the 'bib_gourmand' & 'total' columns
arrondissement_grouped_top10 = arrondissement_grouped_top10.drop(columns=['green_stars', 'selected', 'bib_gourmand', 'total_starred'])

# Reverse the order of the DataFrame
arrondissement_grouped_top10 = arrondissement_grouped_top10.iloc[::-1]

Plot the data

In [ ]:
# Create a horizontal stacked bar plot
arrondissement_grouped_top10.plot(kind='barh', stacked=True, figsize=(10, 7),
                    color=['#fee8c8','#fdbb84','#e34a33'])

# Add title and labels
plt.title('Michelin Starred Restaurants by Arrondissement\n(Excludes Paris)')
plt.ylabel('Arrondissement')
plt.xlabel('Number of Restaurants')

# Add a legend
plt.legend(title='Michelin Stars', labels=['1 Star', '2 Star', '3 Star'])

# Show the plot
plt.show()

----
&nbsp;
#### We now cascade merges from the smallest granularity (arrondissement) up to the highest (region) using the department as an intermediary linking point.

#### 1. Extract departmental code from `geoJSON_df`

In [ ]:
geoJSON_df['department_num'] = geoJSON_df['code'].str[:2]
print(f"Shape: {geoJSON_df.shape}")
print(f"Columns: {geoJSON_df.columns.tolist()}")
geoJSON_df.head(3)

#### 2. Extract department name from file produced in `France_Departments_Regions.ipynb`

In [ ]:
departments = gpd.read_file(DEPARTMENTS_GEOMETRY_PATH)
print(f"Shape: {departments.shape}")
print(f"Columns: {departments.columns.tolist()}")
departments.head()

In [ ]:
# Merge the dataframes on the appropriate columns
merged_geo_df = pd.merge(geoJSON_df[['code', 'nom', 'geometry', 'department_num']],
                         departments[['code', 'nom']],
                         left_on='department_num', right_on='code',
                         how='left', suffixes=('_arrondissement', '_department'))

In [ ]:
print(f"Shape: {merged_geo_df.shape}")

In [ ]:
print(f"Columns: {merged_geo_df.columns.tolist()}")

In [ ]:
# Drop 'code_department'
merged_geo_df.drop('code_department', axis=1, inplace=True)

# Rename columns
column_rename_map = {
    'nom_arrondissement': 'arrondissement',
    'nom_department': 'department'
}
merged_geo_df.rename(columns=column_rename_map, inplace=True)

# Rearrange columns
new_order = ['code_arrondissement', 'arrondissement', 'department_num', 'department', 'geometry']
merged_geo_df = merged_geo_df[new_order]
merged_geo_df.head()

#### 3. Extract the region from `france_df`

In [ ]:
# 1. Create a copy of france_df and retain only the columns of interest
france_subset = france_df[['department_num', 'department', 'capital', 'region']].copy()

# Drop duplicates based on 'department_num'
france_subset = france_subset.drop_duplicates(subset='department_num')
print(f"Shape: {france_subset.shape}")

We know there are 96 departments. Therefore, *all good!*

In [ ]:
# Sort by 'department_num'
france_subset = france_subset.sort_values(by='department_num').reset_index(drop=True)

In [ ]:
print(f"Shape: {france_subset.shape}")
france_subset.head()

In [ ]:
france_subset.info()

#### 4. Merge `france_subset` with `geoJSON_df` on `department_num`

In [ ]:
merged_df = pd.merge(france_subset, geoJSON_df, on="department_num", how="inner")
print(f"Shape geo: {geoJSON_df.shape}\nShape merged: {merged_df.shape}")

In [ ]:
merged_df.head()

#### 5. Merge above with `stats_locale` on 'nom' & 'arrondissement'

In [ ]:
print(f"Shape: {stats_locale.shape}")
stats_locale.head()

In [ ]:
geo_stats = pd.merge(stats_locale, merged_df, left_on='arrondissement', right_on='nom', how='inner')

In [ ]:
print(f"Shape: {geo_stats.shape}")
print(f"Unique regions: {geo_stats['region'].nunique()}")
print(f"Unique departments: {geo_stats['department_num'].nunique()}")
print(f"Unique arrondissements: {geo_stats['arrondissement'].nunique()}")
print(f"Columns:\n{geo_stats.columns.tolist()}")

In [ ]:
# Drop 'code_'
geo_stats.drop(['code_x'], axis=1, inplace=True)
geo_stats.rename(columns={'code_y': 'code'}, inplace=True)

In [ ]:
# Rearrange columns
new_order = ['code', 'arrondissement', 'department_num', 'department', 'capital', 'region', 'municipal_population', 'population_density(inhabitants/sq_km)', 'poverty_rate(%)', 'average_net_hourly_wage(€)', 'geometry']
geo_stats = geo_stats[new_order]
geo_stats.head()

----
&nbsp;
## Merging Michelin Star by *arrondissement* with `geo_stats`

In [ ]:
print(f"Rows in `geo_stats`: {len(geo_stats)}")

# We index the multi-indexed df
arrondissement_grouped.reset_index(inplace=True)
print(f"Rows in `arrondissement_grouped`: {len(arrondissement_grouped)}")

In [ ]:
set1 = set(arrondissement_grouped['arrondissement'].unique())
set2 = set(geo_stats['arrondissement'].unique())
print(f"Sets equal? {set1 == set2}")

print(f"\nIn arrondissement_grouped but not in geo_stats:\n{set1 - set2}")
print(f"\nIn geostats but not in arrondissement_grouped:\n{set2 - set1}")

There are a number of *arrondissement* without Michelin rated restaurants.

These will be written with zero values

In [ ]:
print(f"Columns: {arrondissement_grouped.columns.tolist()}")
arrondissement_grouped.head()

In [ ]:
# Extracting the missing arrondissements
missing_arrondissements = list(set2 - set1)

# Prepare a list to collect new rows
new_rows = []

# Iterate over the missing arrondissements
for arrondissement in missing_arrondissements:
    new_row = {
        'arrondissement': arrondissement,
        'green_stars': 0,
        'selected': 0,
        'bib_gourmand': 0,
        '1_star': 0,
        '2_star': 0,
        '3_star': 0,
        'total_stars': 0,
        'starred_restaurants': 0
    }
    new_rows.append(new_row)

# Create a DataFrame from the new rows and concatenate with the original DataFrame
df_new_rows = pd.DataFrame(new_rows)
arrondissement_grouped = pd.concat([arrondissement_grouped, df_new_rows], ignore_index=True)

# Sorting by 'arrondissement' for better clarity
arrondissement_grouped = arrondissement_grouped.sort_values('arrondissement').reset_index(drop=True)

In [ ]:
print(f"Shape: {arrondissement_grouped.shape}")
arrondissement_grouped.head()

In [ ]:
set1 = set(arrondissement_grouped['arrondissement'].unique())
set2 = set(geo_stats['arrondissement'].unique())
print(f"Sets equal? {set1 == set2}")

----
&nbsp;
## Merging `geo_stats` and `arrondissement_grouped` on 'arrondissement'

In [ ]:
merged_data = pd.merge(geo_stats, arrondissement_grouped, on='arrondissement', how='inner')
print(f"Shape: {merged_data.shape}")
print(f"Columns:\n{merged_data.columns.tolist()}")

In [ ]:
new_order = ['code', 'arrondissement', 'department_num', 'department', 'capital', 'region', 'selected', 'bib_gourmand', '1_star', '2_star', '3_star', 'total_stars', 'starred_restaurants', 'green_stars', 'municipal_population', 'population_density(inhabitants/sq_km)', 'poverty_rate(%)', 'average_net_hourly_wage(€)', 'geometry']
merged_data = merged_data[new_order]
merged_data.head()

----
&nbsp;
## Adding coordinates of individual restaurants

In [ ]:
print(france_df.columns.tolist())

In [ ]:
# Create a separate DataFrame with star ratings, regions, and coordinates
location_data_arrond = france_df[['stars', 'arrondissement', 'latitude', 'longitude']]

# Convert to a dictionary where keys are tuples of star rating and arrondissement
location_dict_arrond = (
    location_data_arrond
    .groupby(['stars', 'arrondissement'])[['latitude', 'longitude']]
    .apply(lambda df: list(zip(df.latitude, df.longitude)))
    .to_dict()
)

In [ ]:
# Now create a function to map these dictionaries to original DataFrame
def map_locations_arrond(row):
    return {stars: location_dict_arrond.get((stars, row['arrondissement'])) for stars in [1, 2, 3]}

In [ ]:
merged_data['locations'] = merged_data.apply(map_locations_arrond, axis=1)

In [ ]:
merged_data.head()

----
&nbsp;
Confirming datatypes

In [ ]:
print(merged_data.dtypes)

In [ ]:
cols_to_convert = [
    'municipal_population',
    'population_density(inhabitants/sq_km)',
    'poverty_rate(%)',
    'average_net_hourly_wage(€)'
]

for col in cols_to_convert:
    merged_data[col] = (
        merged_data[col]
        .astype(str)
        .str.replace(r'[^\d,.\-]', '', regex=True)   # remove non-numeric, non-dot/comma/hyphen chars
        .str.replace(',', '.', regex=False)          # convert comma to decimal point
        .astype(float)
    )

In [ ]:
print(merged_data.dtypes)

----
Export as GeoJSON file

In [ ]:
# Export the GeoDataFrame to a .geojson file
geo_merged_data = gpd.GeoDataFrame(merged_data, geometry='geometry')
save_geojson(geo_merged_data, "arrondissement_restaurants.geojson")